# Wav-KAN-SEUNet - Train với B-Spline KAN (tối ưu tốc độ)

Notebook này sẽ clone mã nguồn từ branch hiện tại, cài đặt các thư viện cần thiết và huấn luyện mô hình với cấu hình B-Spline đã được tối ưu để tránh chậm như đã gặp.

**Cấu hình khuyến nghị:**
- `kan_type = bspline`
- `bspline_grid = 3`
- `bspline_order = 3`
- `batch_size = 2`
- `torch.compile = True`
- Sử dụng AMP + `cudnn.benchmark`

## 1. Clone repo từ branch hiện tại

In [ ]:
import os, sys

REPO_URL = "https://github.com/nghiata-uit/Wav-KAN-SEUNet.git"
BRANCH = "kan-v2"
WORK_DIR = "/content/Wav-KAN-SEUNet"

if os.path.exists(WORK_DIR):
    print(f"Đã tồn tại {WORK_DIR}, cập nhật repo...")
    os.chdir(WORK_DIR)
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"Clone repo {REPO_URL} branch {BRANCH}...")
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
    os.chdir(WORK_DIR)

print("Thư mục hiện tại:", os.getcwd())

## 2. Cài đặt thư viện cần thiết

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q tqdm pandas numpy scipy scikit-learn sympy
!pip install -q pesq pystoi
!pip install -q git+https://github.com/schmiph2/pysepm.git
!pip install -q git+https://github.com/KindXiaoming/pykan.git
!pip install -q ptflops

## 3. Kiểm tra môi trường

In [ ]:
import torch
from kan import KAN

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("KAN import OK:", KAN)

## 4. Cấu hình đường dẫn dữ liệu và tham số train

Hãy chỉnh sửa các đường dẫn dưới đây cho phù hợp với dữ liệu của bạn (VoiceBank-DEMAND hoặc tương tự).

In [ ]:
# ===== CẤU HÌNH DỮ LIỆU =====
CLEAN_DIR = "/content/data/clean_trainset_28spk_wav"
NOISY_DIR = "/content/data/noisy_trainset_28spk_wav"
VAL_CLEAN_DIR = "/content/data/clean_testset_wav"
VAL_NOISY_DIR = "/content/data/noisy_testset_wav"

# ===== CẤU HÌNH HUẤN LUYỆN VỚI B-SPLINE (TỐI ƯU) =====
KAN_TYPE = "bspline"
BSPLINE_GRID = 3
BSPLINE_ORDER = 3
BATCH_SIZE = 2
EPOCHS = 100
LR = 0.001
PATIENCE = 10

## 5. Huấn luyện mô hình

In [ ]:
!python train.py \
  --kan_type {KAN_TYPE} \
  --bspline_grid {BSPLINE_GRID} \
  --bspline_order {BSPLINE_ORDER} \
  --batch_size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --lr {LR} \
  --patience {PATIENCE} \
  --clean_dir {CLEAN_DIR} \
  --noisy_dir {NOISY_DIR} \
  --val_clean_dir {VAL_CLEAN_DIR} \
  --val_noisy_dir {VAL_NOISY_DIR} \
  --compile